# 第5章 回归模型 · 限时测试



本册为依据课程材料新编的训练题，不是教师公布的考试原题。

## 材料依据
- [[课程/深度学习/PPT/Chapter5 (1)/Chapter5 (1).md]]
- [[课程/深度学习/PPT/Chapter2/Chapter2.md]]（仅误差与泛化差距口径）
- [[课程/深度学习/演示代码/Chapter5.py]]
- [[课程/深度学习/课堂笔记/2026-09-23-第8节-深度学习模型训练与正则化讲座.md]]
- [[课程/深度学习/课堂笔记/2026-10-12-第11节-深度学习编程与分类模型宣讲.md]]
- [[课程/深度学习/作业/Homework03/Homework03.md]]
- [[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]]


## 模拟要求

计时 **30 分钟，100 分**；一道完整编程题、多个小问，**从导入库开始独立编写，不做代码补全**。先作答，再运行自检。题面中的接口名称仅用于统一自检，不提供实现框架。

本地 PNG 写入当前目录 `output/`（请用 pathlib 创建目录）；正式考试若题目指定 `/output/Qxxx.png`，须严格使用正式路径和文件名。练习中的输出目录是本地适配，不是正式考试环境的改动。


## Q5B 30分钟综合题：带噪传感器校准与最佳快照

**新编迁移题，依据** [[课程/深度学习/课堂笔记/2026-09-23-第8节-深度学习模型训练与正则化讲座.md#五、显式正则化]]、[[课程/深度学习/课堂笔记/2026-09-23-第8节-深度学习模型训练与正则化讲座.md#六、频谱偏置与早停]]，不同于A卷的无噪声五维响应场。

**共100分，30分钟，从空白独立编程。**

1. **数据（20分）**：CPU单线程，torch.manual_seed(52)。实现`target_b5(x)=sin(2*x[:,0])+.3*x[:,0]`。顺序生成128/64/96条一维均匀[-1,1]训练/验证/测试输入，命名Xb_train/Xb_val/Xb_test；训练标签yb_train加.12*randn噪声，yb_val/yb_test为干净目标。
2. **模型与目标（20分）**：net_b5为1→24→1 Tanh网络；SGD(.05)。实现`l2_b5(net)`，含权重及偏置的平方和。每步优化数据MSE+.001*L2，不能用weight_decay重复惩罚。
3. **训练与早停（30分）**：至多120步；每步更新后重新评估train data MSE、objective、validation MSE，依次存history_b5字典列表，键data_mse/objective/val_mse。验证MSE严格降低即clone保存最佳state_dict与1起始best_epoch_b5，否则未改善计数+1；连续12次未改善停。预算结束也要恢复最佳模型。**测试集不参与决定停止**。
4. **评估（15分）**：恢复后输出report_b5的train_mse/train_rmse/test_rmse/gap_abs，以及best_val_b5。允许未触发早停；记录len(history_b5)为实际更新数。训练误差相对有噪声训练标签计算。
5. **图与解释（15分）**：保存output/Q5B.png，横轴Update，画更新后训练data MSE与validation MSE两线，标记best epoch；不能混画objective冒充训练MSE。写一句为何验证用于早停而测试用于最终报告。

注意：参数快照必须独立于后续训练；本卷不做庞大调参。


In [ ]:
# 从导入库开始，在此独立完成整道编程题。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
import numpy as np
import torch
from pathlib import Path
try:
    assert Xb_train.shape == (128,1) and Xb_val.shape == (64,1) and Xb_test.shape == (96,1)
    assert torch.allclose(target_b5(torch.zeros(1,1)), torch.zeros(1))
    assert net_b5(torch.zeros(1,1)).shape == (1,1)
    assert sum(p.numel() for p in net_b5.parameters()) == 73
    assert 1 <= best_epoch_b5 <= len(history_b5) <= 120
    assert all(r["objective"] >= r["data_mse"] for r in history_b5)
    assert np.isclose(best_val_b5, min(r["val_mse"] for r in history_b5))
    with torch.no_grad():
        restored_b5 = (net_b5(Xb_val).squeeze(-1)-yb_val).square().mean().item()
    assert np.isclose(restored_b5, best_val_b5, rtol=1e-5, atol=1e-8)
    assert np.isfinite(list(report_b5.values())).all()
    assert np.isclose(report_b5["gap_abs"], abs(report_b5["test_rmse"]-report_b5["train_rmse"]))
    assert Path("output/Q5B.png").stat().st_size > 1000
except (NameError, NotImplementedError) as exc:
    print("尚未完成本题：", exc)
else:
    print("自检通过；仍需人工核对题目全部要求。")
